# <span style="font-family:cursive;">Predicting Heart Disease using Machine Learning ❤</span>

![](https://www.mydr.com.au/wp-content/uploads/2019/04/heart_failure_750.jpeg)

**<span style="font-family:cursive;">Machine Learning is used across many spheres around the world. The healthcare industry is no exception. Machine Learning can play an essential role in predicting presence/absence of Locomotor disorders, Heart diseases and more. Such information, if predicted well in advance, can provide important insights to doctors who can then adapt their diagnosis and treatment per patient basis.</span>**

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#5642C5;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">

<p style="padding: 25px; color:white; text-align:center"><b>Importing Libraries</b></p>
</div>

In [ ]:
import numpy as np 
import pandas as pd 

# Libraries for visualisation
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('fivethirtyeight')

# Libraries for Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer

# Machine Learning Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Libraries for Metrics
from sklearn.metrics import accuracy_score, confusion_matrix

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Reading the data set
data = pd.read_csv('/kaggle/input/heart-failure-prediction/heart.csv')
print("Number of rows in the dataset: {}".format(data.shape[0]))
print("Number of cols in the dataset: {}".format(data.shape[1]))


In [ ]:
data.head()

**📌 Attribute Information:**
* **Age:** age of the patient [years]
* **Sex:** sex of the patient [M: Male, F: Female]
* **ChestPainType:** chest pain type [TA: Typical Angina, ATA: Atypical Angina, NAP: Non-Anginal Pain, ASY: Asymptomatic]
* **RestingBP:** resting blood pressure [mm Hg]
* **Cholesterol:** serum cholesterol [mm/dl]
* **FastingBS:** fasting blood sugar [1: if FastingBS > 120 mg/dl, 0: otherwise]
* **RestingECG:** resting electrocardiogram results [Normal: Normal, ST: having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV), LVH: showing probable or definite left ventricular hypertrophy by Estes' criteria]
* **MaxHR:** maximum heart rate achieved [Numeric value between 60 and 202]
* **ExerciseAngina:** exercise-induced angina [Y: Yes, N: No]
* **Oldpeak:** oldpeak = ST [Numeric value measured in depression]
* **ST_Slope:** the slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: downsloping]
* **HeartDisease:** output class [1: heart disease, 0: Normal]

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#5642C5;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">

<p style="padding: 25px; color:white; text-align:center"><b>🔍   Exploratory Data Analysis</b></p>
</div>

In [ ]:
data.isnull().sum()

<div class="alert alert-block alert-info" style="text-align:center"> 📌<b>Insights:</b> We can cearly see that there is no NaN values (missing values) in the dataset.</div>

In [ ]:
data.describe().T.style.background_gradient(cmap='Blues')

**Cardinality of the data**

In [ ]:
color = plt.cm.copper(np.linspace(0, 1, 10))
plt.figure(figsize=(14,5))
data.nunique().plot(kind='bar', width=.8, color=color)
plt.title('No of unique values in the dataset', size=16)
plt.show()
print(data.nunique())

<div class="alert alert-block alert-info" style="text-align:center"> 📌<b>Insights: </b>Here we have different types of data, such as categorical, numerical (discrette and continuous)<br>categorical features: Sex, Chest Pain Type, Resting ECG, Exercise Angina, ST_Slope<br>Numerical features:
    Age, RestingBP, cholesterol, FastingBS, MaxHR, Oldpeak, Heart Disease(Target)<br>Discrette Features: Sex, ChestPainType, FastingBS, RestingECG, ExerciseAngina, ST_Slope<br> Continuous features: Age, RestingBP, Cholesterol, MaxHR, Oldpeak
</div>

**Seperating the data into discrette features and continuous feat**

In [ ]:
discrette_feature = [i for i in data.columns if data[i].nunique() < 10]
continuous_feature = [i for i in data.columns if data[i].nunique() > 10]
print(f'discrette feature: {discrette_feature}')
print(f'continuous feature: {continuous_feature}')

**Visualising discrette feature with output variable count plot**

In [ ]:
plt.style.use(plt.style.available[19])
i = 1
plt.figure(figsize=(18,20)) 
for feature in discrette_feature[:-1]:
    plt.subplot(4, 2, i)
    sns.countplot(x=data[feature], hue=data.HeartDisease)
    i += 1
    
plt.show()

<div class="alert alert-block alert-info" style="text-align:center"> 📌<b>Insights: </b>Sex: Male is having more heart disease compared to female<br>Chest pain type ASY: Asymptomatic having more heart disease compared to others<br>exercise-induced angina: yes and ST_Slope: Flat is having more compared to others<br>And peoples with ST_Slope: up, ExerciseAngina: No and chest pain type: ATA (ATypical angina) mostly don't have heart disease</div>

**visualising continuous features with output variable using histogram and kde**

In [ ]:
plt.style.use('fivethirtyeight')
i = 1
plt.figure(figsize=(18,16)) 
for feature in continuous_feature:
    plt.subplot(3, 2, i)
    sns.histplot(x=data[feature],kde=True,bins=50, hue=data.HeartDisease)
    plt.xlabel(feature,size=12)
    plt.ylabel("Density",size=12)
    i += 1
    
plt.show()

<div class="alert alert-block alert-info" style="text-align:center"> 📌<b>Insights :</b> people who are aged more than 50 and having MaxHR (maximum Heart Rate) less than 140 and also have more Oldpeak is more likely to have heart disease.</div>

**visualising continuous features with each other using scatter plot**

In [ ]:
i = 1
n = 1
plt.figure(figsize=(18,24)) 
for feature in continuous_feature:
    for i_ in continuous_feature[n:]:
        plt.subplot(5, 2, i)
        sns.scatterplot(x=data[feature],y=data[i_], s=80, hue=data.HeartDisease)
        plt.xlabel(feature,size=12)
        plt.ylabel(i_,size=12)
        i += 1
    n+=1
plt.show()        

In [ ]:
plt.style.use('fivethirtyeight')
i = 1
plt.figure(figsize=(18,16)) 
for feature in continuous_feature:
    plt.subplot(3, 2, i)
    sns.boxplot(data=data, x=feature, hue='HeartDisease', color='darkcyan')
    plt.xlabel(feature,size=12)
#     plt.ylabel("Density",size=12)
    i += 1
    
plt.show()

In [ ]:
plt.figure(figsize=(14,8))
sns.heatmap(data.corr(), mask=np.triu(data.corr()), annot=True, cmap='YlGnBu',linewidth=2)
plt.show()

<div class="alert alert-block alert-info" style="text-align:center"> 📌<b>Insights :</b> Oldpeak and Heart Disease have strong positive correlation, whereas maxHR (maximum Heart Rate) and Age have negative correlation (younger people have more heart rate compared to older people)</div>

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#5642C5;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">

<p style="padding: 25px; color:white; text-align:center"><b>Data Preprocessing</b></p>
</div>

In [ ]:
data.head()

**seperating input features and output features**

In [ ]:
X = data.drop(columns=['HeartDisease'])
y = data['HeartDisease']

**splitting the data for training and testing**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=25)
print(f'x_train shape: {x_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'x_test shape : {x_test.shape}')
print(f'y_test shape : {y_test.shape}')

**Applying One Hot Encoding and standardization using column transformer**

In [ ]:
transformer = ColumnTransformer(transformers = [('onehot', OneHotEncoder(sparse=False, drop='first'), [1,2,6,8,10]),
                                                ('scaler', StandardScaler(), [0,3,4,5,7,9])],
                                      remainder = 'passthrough')
x_train = transformer.fit_transform(x_train)
x_test = transformer.transform(x_test)

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#5642C5;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">

<p style="padding: 25px; color:white; text-align:center"><b> Model Building</b></p>
</div>

In [ ]:
def do_prediction(classifier):
    
    # training the classifier on the dataset
    classifier.fit(x_train, y_train)
    
    #Do prediction and evaluting the prediction
    prediction = classifier.predict(x_test)
    cross_validation_score = cross_val(x_train,y_train, classifier)
    accuracy = accuracy_score(y_test, prediction)
    con_metrix = confusion_matrix(y_test, prediction)
    
    return accuracy,con_metrix, cross_validation_score

def cross_val(x_train, y_train, classifier):
    
    # Applying k-Fold Cross Validation
    accuracies = cross_val_score(estimator = classifier, X = x_train, y = y_train, cv = 5)
    return accuracies.mean()


**Logistic Regression**

In [ ]:

logistic_reg = LogisticRegression()
accuracy,con_metrix, cross_validation_score = do_prediction(logistic_reg)

plt.figure(figsize=(9,6))
sns.heatmap(con_metrix, annot=True, cmap='PuBuGn')
plt.title('Confusion Matrix for logistic regression',size=14)
plt.show()
print('Logistic Regression Performace on the training data have an accuracy score of {}'.format((cross_validation_score *100).round()) )
print('Logistic Regression Performace on the testing data have an accuracy score of {}'.format((accuracy *100).round()) )

**Naive Bayes**

In [ ]:
gaussian = GaussianNB()
accuracy,con_metrix, cross_validation_score = do_prediction(gaussian)

plt.figure(figsize=(9,6))
sns.heatmap(con_metrix, annot=True, cmap='PuBuGn')
plt.title('Confusion Matrix for Naive Bayes',size=14)
plt.show()
print('Naive Bayes Performace on the training data have an accuracy score of {}'.format((cross_validation_score *100).round()) )
print('Naive Bayes Performace on the testing data have an accuracy score of {}'.format((accuracy *100).round()) )

**Support Vector Machine**

In [ ]:
svm = SVC()
accuracy,con_metrix, cross_validation_score = do_prediction(svm)

plt.figure(figsize=(9,6))
sns.heatmap(con_metrix, annot=True, cmap='PuBuGn')
plt.title('Confusion Matrix for SVM',size=14)
plt.show()
print('Support Vector Machine Performace on the training data have an accuracy score of {}'.format((cross_validation_score *100).round()) )
print('Support Vector Machine Performace on the testing data have an accuracy score of {}'.format((accuracy *100).round()) )